# Week 6: Channel-Based Sampling — TED and TEDx

This notebook initiates a pivot to channel-based sampling by collecting all videos
from the official TED and TEDx YouTube channels. The goal is to construct a
systematic dataset of expert discourse on YouTube while reusing the sentiment
analysis pipeline developed in earlier weeks.


In [7]:
!git clone https://github.com/mustafayubk/SOSC314_Project.git
%cd SOSC314_Project
!ls


Cloning into 'SOSC314_Project'...
remote: Enumerating objects: 377, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 377 (delta 102), reused 0 (delta 0), pack-reused 199 (from 2)
Receiving objects: 100% (377/377), 9.76 MiB | 15.99 MiB/s, done.
Resolving deltas: 100% (202/202), done.
/content/SOSC314_Project/SOSC314_Project
 config   docs	      README.md  'Week 2_Figure.png'
 data	  notebooks   scripts	  Week_3_Figure.png


In [8]:
import os

os.makedirs("notebooks/week6", exist_ok=True)


## Step 1: Retrieve all videos from TED and TEDx channels

This step uses the YouTube Data API to collect metadata for all videos
uploaded by the official TED and TEDx channels. This provides a systematic,
channel-based sampling frame for expert discourse on YouTube.


In [9]:
!pip -q install google-api-python-client pandas tqdm


In [10]:
import os
import pandas as pd
from tqdm import tqdm
from googleapiclient.discovery import build


In [11]:
from google.colab import userdata


In [12]:
API_KEY = userdata.get("YOUTUBE_API_KEY") or userdata.get("YOUTUBE")

print("Has key?", API_KEY is not None)

if not API_KEY:
    raise ValueError("YouTube API key not found. In Colab Secrets, create YOUTUBE_API_KEY (or YOUTUBE) and enable Notebook access.")

youtube = build("youtube", "v3", developerKey=API_KEY)
print("YouTube client built ✅")


Has key? True
YouTube client built ✅


In [13]:
CHANNELS = {
    "TED": "UCAuUUnT6oDeKwE6v1NGQxug",
    "TEDx": "UCsT0YIqwnpJCM-mx7-gSA4Q"
}


In [14]:
def get_all_videos_from_channel(channel_id, channel_name):
    videos = []
    next_page = None

    while True:
        request = youtube.search().list(
            part="snippet",
            channelId=channel_id,
            maxResults=50,
            pageToken=next_page,
            type="video",
            order="date"
        )
        response = request.execute()

        for item in response["items"]:
            videos.append({
                "video_id": item["id"]["videoId"],
                "title": item["snippet"]["title"],
                "channel": channel_name,
                "published_at": item["snippet"]["publishedAt"]
            })

        next_page = response.get("nextPageToken")
        if not next_page:
            break

    return videos


In [15]:
all_videos = []

for name, cid in CHANNELS.items():
    print(f"Collecting videos from {name}...")
    vids = get_all_videos_from_channel(cid, name)
    print(f"  → {len(vids)} videos found")
    all_videos.extend(vids)


  → 9 videos found
  → 100 videos found


In [16]:
def get_uploads_playlist_id(channel_id: str) -> str:
    resp = youtube.channels().list(
        part="contentDetails",
        id=channel_id
    ).execute()
    items = resp.get("items", [])
    if not items:
        raise ValueError(f"No channel found for channel_id={channel_id}")
    return items[0]["contentDetails"]["relatedPlaylists"]["uploads"]

for name, cid in CHANNELS.items():
    upl = get_uploads_playlist_id(cid)
    print(name, "uploads playlist:", upl)


TED uploads playlist: UUAuUUnT6oDeKwE6v1NGQxug
TEDx uploads playlist: UUsT0YIqwnpJCM-mx7-gSA4Q


In [17]:
def get_all_videos_from_uploads_playlist(uploads_playlist_id: str, channel_name: str):
    videos = []
    page_token = None

    while True:
        resp = youtube.playlistItems().list(
            part="snippet,contentDetails",
            playlistId=uploads_playlist_id,
            maxResults=50,
            pageToken=page_token
        ).execute()

        for item in resp.get("items", []):
            # playlistItems gives videoId + publishedAt inside contentDetails
            vid = item["contentDetails"]["videoId"]
            published_at = item["contentDetails"].get("videoPublishedAt") or item["snippet"].get("publishedAt")
            title = item["snippet"]["title"]

            videos.append({
                "video_id": vid,
                "title": title,
                "channel": channel_name,
                "published_at": published_at
            })

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    return videos

all_videos = []
for name, cid in CHANNELS.items():
    uploads = get_uploads_playlist_id(cid)
    print(f"Collecting ALL uploads from {name}…")
    vids = get_all_videos_from_uploads_playlist(uploads, name)
    print(f"  → {len(vids)} videos found")
    all_videos.extend(vids)

print("TOTAL videos:", len(all_videos))


  → 5491 videos found
  → 20000 videos found
TOTAL videos: 25491


In [18]:
import pandas as pd

videos_df = pd.DataFrame(all_videos)
videos_df["published_at"] = pd.to_datetime(videos_df["published_at"], errors="coerce")

print(videos_df.groupby("channel")["video_id"].nunique())
print("Earliest by channel:")
print(videos_df.sort_values("published_at").groupby("channel").head(1)[["channel","published_at","title"]])
print("Latest by channel:")
print(videos_df.sort_values("published_at").groupby("channel").tail(1)[["channel","published_at","title"]])


channel
TED      5491
TEDx    20000
Name: video_id, dtype: int64
Earliest by channel:
      channel              published_at  \
5490      TED 2006-12-25 17:58:08+00:00   
25490    TEDx 2025-05-01 15:30:39+00:00   

                                                   title  
5490                If I controlled the Internet | Rives  
25490  Why you should make a ‘done’ list | Francis Ru...  
Latest by channel:
     channel              published_at  \
5491    TEDx 2026-02-08 14:30:11+00:00   
0        TED 2026-02-08 19:00:41+00:00   

                                                  title  
5491  How do non-living things ‘evolve’? | Michael W...  
0        What if plastic didn’t last forever? #TEDTalks  


In [19]:
import os
import pandas as pd

# Create folder if it doesn't exist
os.makedirs("data/raw/week6", exist_ok=True)

videos_df = pd.DataFrame(all_videos)

# (Optional but recommended) remove any duplicates just in case
videos_df = videos_df.drop_duplicates(subset=["video_id"])

out_path = "data/raw/week6/ted_tedx_videos.csv"
videos_df.to_csv(out_path, index=False)

print("Saved:", len(videos_df), "videos to", out_path)
videos_df.head()


Saved: 25491 videos to data/raw/week6/ted_tedx_videos.csv


,video_id,title,channel,published_at
0,iOTCsXd38Ng,What if plastic didn’t last forever? #TEDTalks,TED,2026-02-08T19:00:41Z
1,d1yfb93beSI,How to Introduce Yourself — and Get Hired | Re...,TED,2026-02-08T16:00:53Z
2,yrVtEkCC8uY,These ads could encourage people to think twic...,TED,2026-02-07T19:01:00Z
3,an6ZM0iCGQw,Let’s Build AI Data Centers in Space | Philip ...,TED,2026-02-06T16:00:07Z
4,bA9xGDsQADo,Today’s athletes ARE built different #TEDTalks,TED,2026-02-05T20:00:34Z


In [20]:
import os, glob
print("Folder exists?", os.path.exists("data/raw/week6"))
print("Files:", glob.glob("data/raw/week6/*"))


Folder exists? True
Files: ['data/raw/week6/ted_tedx_videos.csv']


## Week 6: Channel-Based Sampling Frame (TED & TEDx)

This section constructs the full population of TED and TEDx videos
and assigns each video to a publication-period time bin.

All subsequent comment scraping and sentiment analysis will operate
on this fixed channel-level sampling frame to ensure reproducibility
and comparability across time.


In [21]:
# Fix: ensure published_at is a proper datetime (not a string)
videos_df["published_at"] = pd.to_datetime(videos_df["published_at"], errors="coerce")

print("Null dates after conversion:", videos_df["published_at"].isna().sum())
print("Example:", videos_df["published_at"].dropna().iloc[0])


Null dates after conversion: 0
Example: 2026-02-08 19:00:41+00:00


In [22]:
# Define publication-period bins (same logic as earlier weeks)
def assign_time_bin(dt):
    year = dt.year
    if year < 2010:
        return "Pre-2010"
    elif year < 2020:
        return "2010–2019"
    else:
        return "2020+"

videos_df["time_bin"] = videos_df["published_at"].apply(assign_time_bin)

# Quick sanity check
videos_df.groupby(["channel", "time_bin"])["video_id"].nunique()


channel  time_bin 
TED      2010–2019     2620
         2020+         2300
         Pre-2010       571
TEDx     2020+        20000
Name: video_id, dtype: int64

In [23]:
# 3️⃣ Save the finalized sampling frame (Week 6 core artifact)

import os

out_path = "data/processed/week6/ted_tedx_video_sampling_frame.csv"
os.makedirs("data/processed/week6", exist_ok=True)

videos_df.to_csv(out_path, index=False)

print("Saved sampling frame to:", out_path)
print("Rows:", videos_df.shape[0])
print("Columns:", videos_df.columns.tolist())


Saved sampling frame to: data/processed/week6/ted_tedx_video_sampling_frame.csv
Rows: 25491
Columns: ['video_id', 'title', 'channel', 'published_at', 'time_bin']


## Week 6 — Comment Scraping Scaffold (Test Run)

Now that we have a full TED + TEDx video sampling frame, the next step is to scrape
top-level comments for each video.

To keep the pipeline comparable to earlier weeks and manageable under API quota,
I start by scraping **top-level comments only** (no replies) and I cap the number
of comments per video (I will justify this in the Week 6 report).

Before scraping the full dataset, we do a small **test run** on a few videos to confirm:
- the API calls work
- the output format is correct
- comments are actually being collected and saved


In [24]:
import time
import pandas as pd

def get_top_level_comments(video_id, max_comments=200, sleep_s=0.1):
    """
    Scrape up to max_comments top-level comments (no replies) from a YouTube video.
    Returns a list of dicts.
    """
    comments = []
    next_page = None

    while len(comments) < max_comments:
        req = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=min(100, max_comments - len(comments)),
            pageToken=next_page,
            textFormat="plainText",
            order="time"  # consistent ordering
        )
        res = req.execute()

        items = res.get("items", [])
        for it in items:
            sn = it["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "video_id": video_id,
                "comment_id": it["snippet"]["topLevelComment"]["id"],
                "text": sn.get("textDisplay", ""),
                "like_count": sn.get("likeCount", 0),
                "published_at": sn.get("publishedAt", "")
            })

        next_page = res.get("nextPageToken")
        if not next_page:
            break

        time.sleep(sleep_s)

    return comments


# ---------------------------
# TEST RUN (tiny sample)
# ---------------------------
# Pick a few videos from your sampling frame
test_ted = videos_df[videos_df["channel"] == "TED"]["video_id"].head(2).tolist()
test_tedx = videos_df[videos_df["channel"] == "TEDx"]["video_id"].head(2).tolist()
test_ids = test_ted + test_tedx

print("Test video IDs:", test_ids)

all_test_comments = []
for vid in test_ids:
    try:
        batch = get_top_level_comments(vid, max_comments=200)
        print(f"{vid}: collected {len(batch)} comments")
        all_test_comments.extend(batch)
    except Exception as e:
        print(f"{vid}: FAILED -> {e}")

# Save test output
import os
os.makedirs("data/raw/week6", exist_ok=True)

test_df = pd.DataFrame(all_test_comments)
out_path = "data/raw/week6/comments_test_sample.csv"
test_df.to_csv(out_path, index=False)

print("Saved test comments to:", out_path)
test_df.head()


Test video IDs: ['iOTCsXd38Ng', 'd1yfb93beSI', 'DSrf7ErdHWA', 'TEf0EV3mVWw']
iOTCsXd38Ng: collected 10 comments
d1yfb93beSI: collected 37 comments
DSrf7ErdHWA: collected 14 comments
TEf0EV3mVWw: collected 23 comments
Saved test comments to: data/raw/week6/comments_test_sample.csv


,video_id,comment_id,text,like_count,published_at
0,iOTCsXd38Ng,UgxaVM4LSOrH4e0Pamt4AaABAg,Sand and Silica is are among the Most abundant...,1,2026-02-08T23:58:52Z
1,iOTCsXd38Ng,UgywYl7CRxnk1yUoWCR4AaABAg,"Plastic is everywhere and I mean everywhere, ...",0,2026-02-08T23:00:34Z
2,iOTCsXd38Ng,Ugyqmuz52AJOZBUTynd4AaABAg,Brava❤,1,2026-02-08T22:11:44Z
3,iOTCsXd38Ng,UgzB3f0d76GDGn7uL6B4AaABAg,Amazing! My respects.,1,2026-02-08T22:02:43Z
4,iOTCsXd38Ng,UgzLsOVSfvA8jzipqJ54AaABAg,IT IS A LIE! WE KNOW IT. DONT FOOL US ANY MORE...,3,2026-02-08T20:46:49Z


## Week 6 — Full Comment Scrape (Chunked + Resume-Friendly)

We now scrape comments for the full TED + TEDx sampling frame.
To avoid losing progress and to keep files manageable, we save data in chunks.
Each chunk contains comments for a fixed number of videos.

If Colab disconnects or quota is hit, we can resume from the last completed chunk.


In [25]:
import os
import pandas as pd
import time

# ---------- SETTINGS ----------
MAX_COMMENTS_PER_VIDEO = 500     # consistent with earlier weeks
VIDEOS_PER_CHUNK = 25            # adjust later if needed (25 is safe)
SLEEP_BETWEEN_CALLS = 0.1        # tiny delay to be polite

# Save folder
RAW_DIR = "data/raw/week6"
os.makedirs(RAW_DIR, exist_ok=True)

# Load sampling frame (from your committed CSV)
frame_path = "data/processed/week6/ted_tedx_video_sampling_frame.csv"
videos_df = pd.read_csv(frame_path)

# Safety: ensure correct columns exist
needed_cols = {"video_id", "channel", "published_at", "time_bin"}
missing = needed_cols - set(videos_df.columns)
if missing:
    raise ValueError(f"Sampling frame missing columns: {missing}")

# ---------- RESUME LOGIC ----------
# We'll store a small progress file that tracks which videos are already done
progress_path = f"{RAW_DIR}/scrape_progress.csv"

if os.path.exists(progress_path):
    progress_df = pd.read_csv(progress_path)
    done_ids = set(progress_df["video_id"].astype(str))
    print("Resuming: already scraped videos =", len(done_ids))
else:
    done_ids = set()
    progress_df = pd.DataFrame(columns=["video_id", "status", "n_comments"])
    print("Starting fresh scrape")

# Filter videos that still need scraping
todo = videos_df[~videos_df["video_id"].astype(str).isin(done_ids)].copy()
todo_ids = todo["video_id"].astype(str).tolist()

print("Videos remaining to scrape:", len(todo_ids))

# ---------- SCRAPE 1 CHUNK ONLY (for Tuesday Commit 1) ----------
# We only scrape the FIRST chunk today to prove full pipeline works.
chunk_ids = todo_ids[:VIDEOS_PER_CHUNK]
print("Scraping this chunk size:", len(chunk_ids))

all_comments = []
new_progress_rows = []

for i, vid in enumerate(chunk_ids, start=1):
    try:
        batch = get_top_level_comments(vid, max_comments=MAX_COMMENTS_PER_VIDEO, sleep_s=SLEEP_BETWEEN_CALLS)
        all_comments.extend(batch)
        new_progress_rows.append({"video_id": vid, "status": "ok", "n_comments": len(batch)})
        print(f"[{i}/{len(chunk_ids)}] {vid}: {len(batch)} comments")
    except Exception as e:
        new_progress_rows.append({"video_id": vid, "status": f"fail: {e}", "n_comments": 0})
        print(f"[{i}/{len(chunk_ids)}] {vid}: FAILED -> {e}")

# Convert and merge metadata (channel + time_bin)
comments_df = pd.DataFrame(all_comments)
comments_df = comments_df.merge(videos_df[["video_id","channel","time_bin"]], on="video_id", how="left")

# Save this chunk
chunk_index = len(done_ids) // VIDEOS_PER_CHUNK
chunk_path = f"{RAW_DIR}/comments_chunk_{chunk_index:03d}.csv"
comments_df.to_csv(chunk_path, index=False)

# Update progress log
progress_df = pd.concat([progress_df, pd.DataFrame(new_progress_rows)], ignore_index=True)
progress_df.to_csv(progress_path, index=False)

print("✅ Saved chunk to:", chunk_path)
print("✅ Updated progress log:", progress_path)
print("Chunk rows:", len(comments_df))
comments_df.head()


Starting fresh scrape
Videos remaining to scrape: 25491
Scraping this chunk size: 25
[1/25] iOTCsXd38Ng: 10 comments
[2/25] d1yfb93beSI: 37 comments
[3/25] yrVtEkCC8uY: 4 comments
[4/25] an6ZM0iCGQw: 79 comments
[5/25] bA9xGDsQADo: 6 comments
[6/25] kryWR7QNBkc: 29 comments
[7/25] c7rtbTnmnzI: 13 comments
[8/25] HtzG2AoahAo: 23 comments
[9/25] -CAzAfDhbwU: 9 comments
[10/25] vkHuZY-x5DA: 21 comments
[11/25] 7IYKojMhBy4: 27 comments
[12/25] roc73qEYhXY: 14 comments
[13/25] X564_fIhp_0: 500 comments
[14/25] VoPDKjFHIDU: 45 comments
[15/25] VFY6NTaw9mM: 45 comments
[16/25] YBH8rQv4aTQ: 53 comments
[17/25] Mqk_j_248MU: 15 comments
[18/25] E74vI_Nrn4k: 9 comments
[19/25] LsM8fyov0iI: 15 comments
[20/25] IuzK04m-Px4: 14 comments
[21/25] VA2hdDoSM4s: 74 comments
[22/25] E6oZzu_ui0E: 11 comments
[23/25] vsMqvHqnRUs: 60 comments
[24/25] aWGu4owdXrM: 14 comments
[25/25] ojttMNOW6zM: 500 comments
✅ Saved chunk to: data/raw/week6/comments_chunk_000.csv
✅ Updated progress log: data/raw/week6/scrape

,video_id,comment_id,text,like_count,published_at,channel,time_bin
0,iOTCsXd38Ng,UgxaVM4LSOrH4e0Pamt4AaABAg,Sand and Silica is are among the Most abundant...,1,2026-02-08T23:58:52Z,TED,2020+
1,iOTCsXd38Ng,UgywYl7CRxnk1yUoWCR4AaABAg,"Plastic is everywhere and I mean everywhere, ...",0,2026-02-08T23:00:34Z,TED,2020+
2,iOTCsXd38Ng,Ugyqmuz52AJOZBUTynd4AaABAg,Brava❤,1,2026-02-08T22:11:44Z,TED,2020+
3,iOTCsXd38Ng,UgzB3f0d76GDGn7uL6B4AaABAg,Amazing! My respects.,1,2026-02-08T22:02:43Z,TED,2020+
4,iOTCsXd38Ng,UgzLsOVSfvA8jzipqJ54AaABAg,IT IS A LIE! WE KNOW IT. DONT FOOL US ANY MORE...,3,2026-02-08T20:46:49Z,TED,2020+


In [26]:
import os
import pandas as pd
import time

# ---------- SETTINGS ----------
MAX_COMMENTS_PER_VIDEO = 500
VIDEOS_PER_CHUNK = 25

# For Tuesday commit 2: run multiple chunks in one go
NUM_CHUNKS_TO_RUN_NOW = 4   # 4 chunks × 25 vids = 100 videos today (safe)
SLEEP_BETWEEN_CALLS = 0.1

RAW_DIR = "data/raw/week6"
os.makedirs(RAW_DIR, exist_ok=True)

frame_path = "data/processed/week6/ted_tedx_video_sampling_frame.csv"
videos_df = pd.read_csv(frame_path)

needed_cols = {"video_id", "channel", "published_at", "time_bin"}
missing = needed_cols - set(videos_df.columns)
if missing:
    raise ValueError(f"Sampling frame missing columns: {missing}")

progress_path = f"{RAW_DIR}/scrape_progress.csv"

if os.path.exists(progress_path):
    progress_df = pd.read_csv(progress_path)
    done_ids = set(progress_df["video_id"].astype(str))
    print("Resuming: already scraped videos =", len(done_ids))
else:
    progress_df = pd.DataFrame(columns=["video_id", "status", "n_comments"])
    done_ids = set()
    print("Starting fresh scrape")

todo = videos_df[~videos_df["video_id"].astype(str).isin(done_ids)].copy()
todo_ids = todo["video_id"].astype(str).tolist()

print("Videos remaining to scrape:", len(todo_ids))

# Determine how many videos we will scrape this run
total_to_scrape = min(len(todo_ids), NUM_CHUNKS_TO_RUN_NOW * VIDEOS_PER_CHUNK)
run_ids = todo_ids[:total_to_scrape]

print("Scraping videos this run:", total_to_scrape)
print("Expected new chunk files:", (total_to_scrape + VIDEOS_PER_CHUNK - 1) // VIDEOS_PER_CHUNK)

# ---------- SCRAPE LOOP ----------
new_progress_rows = []
all_chunk_paths = []

for chunk_start in range(0, total_to_scrape, VIDEOS_PER_CHUNK):
    chunk_ids = run_ids[chunk_start:chunk_start + VIDEOS_PER_CHUNK]
    all_comments = []

    # compute chunk index based on how many are already done
    chunk_index = (len(done_ids) + chunk_start) // VIDEOS_PER_CHUNK
    print(f"\n--- Chunk {chunk_index:03d} | videos {chunk_start+1}-{chunk_start+len(chunk_ids)} ---")

    for i, vid in enumerate(chunk_ids, start=1):
        try:
            batch = get_top_level_comments(vid, max_comments=MAX_COMMENTS_PER_VIDEO, sleep_s=SLEEP_BETWEEN_CALLS)
            all_comments.extend(batch)
            new_progress_rows.append({"video_id": vid, "status": "ok", "n_comments": len(batch)})
            print(f"[{i}/{len(chunk_ids)}] {vid}: {len(batch)}")
        except Exception as e:
            new_progress_rows.append({"video_id": vid, "status": f"fail: {e}", "n_comments": 0})
            print(f"[{i}/{len(chunk_ids)}] {vid}: FAILED -> {e}")

    comments_df = pd.DataFrame(all_comments)
    if len(comments_df) > 0:
        comments_df = comments_df.merge(videos_df[["video_id","channel","time_bin"]], on="video_id", how="left")

    chunk_path = f"{RAW_DIR}/comments_chunk_{chunk_index:03d}.csv"
    comments_df.to_csv(chunk_path, index=False)
    all_chunk_paths.append(chunk_path)

    print("✅ Saved:", chunk_path, "| rows:", len(comments_df))

# ---------- UPDATE PROGRESS LOG ----------
progress_df = pd.concat([progress_df, pd.DataFrame(new_progress_rows)], ignore_index=True)
progress_df.to_csv(progress_path, index=False)

print("\n✅ Updated progress log:", progress_path)
print("Total tracked videos:", len(progress_df))

# ---------- SCRAPE SUMMARY (THIS IS IMPORTANT FOR WEEK 6 REPORT) ----------
summary = (
    progress_df.assign(ok=progress_df["status"].eq("ok"))
    .groupby("ok")["video_id"].count()
    .rename(index={True:"ok", False:"failed"})
)
print("\nScrape status counts:")
display(summary)

print("\nComment count distribution (only ok videos):")
display(progress_df[progress_df["status"]=="ok"]["n_comments"].describe())

# Save a summary table for reporting
summary_path = f"{RAW_DIR}/scrape_summary.csv"
progress_df.to_csv(summary_path, index=False)
print("✅ Saved summary table:", summary_path)

print("\nNew chunk files created this run:")
for p in all_chunk_paths:
    print(" -", p)


Resuming: already scraped videos = 25
Videos remaining to scrape: 25466
Scraping videos this run: 100
Expected new chunk files: 4

--- Chunk 001 | videos 1-25 ---
[1/25] nCSexSgcIuI: 11
[2/25] 81slns4Dte8: 39
[3/25] DY2ojllbpFc: 20
[4/25] w-kKNnnqOHk: 10
[5/25] tKO9WgPZ0Bc: 33
[6/25] uqWh9Qt6Wc4: 11
[7/25] ZYIUnmrkmRI: 8
[8/25] C6lTGCAitXg: 15
[9/25] 2-7U8bLjlJE: 9
[10/25] P1rPiSxYagM: 168
[11/25] Kc6cvBFoEzA: 11
[12/25] Bej0gPnHoTY: 19
[13/25] 2pPPJbnZOtQ: 82
[14/25] t5BqFmvHRtc: 12
[15/25] -iGbPbAaqLs: 45
[16/25] K5NEEAr-D6Q: 12
[17/25] 1O9WjWzvz9I: 21
[18/25] 5gm15Q4iSZg: 11
[19/25] wx2Ml3vWHys: 13
[20/25] rh4lXr8JCNQ: 24
[21/25] fStLnjrZF_c: 52
[22/25] dqVfnC_muaI: 187
[23/25] 1E4rBf6oNvM: 34
[24/25] cJfKqKEyw1o: 184
[25/25] ZLq1SbBQRWY: 124
✅ Saved: data/raw/week6/comments_chunk_001.csv | rows: 1155

--- Chunk 002 | videos 26-50 ---
[1/25] RX0ik6nWc58: 3
[2/25] L2OVDJ6cDJc: 104
[3/25] YrZqeJLJzpM: 28
[4/25] 3bbSk3899Po: 26
[5/25] _ozdlc2ouYc: 16
[6/25] gw0UK24zFxw: 7
[7/25] 0R_CJj

,video_id
ok,
ok,125



Comment count distribution (only ok videos):


,n_comments
count,125.000000
mean,53.816000
std,89.326096
min,1.000000
25%,13.000000
50%,24.000000
75%,51.000000
max,500.000000


✅ Saved summary table: data/raw/week6/scrape_summary.csv

New chunk files created this run:
 - data/raw/week6/comments_chunk_001.csv
 - data/raw/week6/comments_chunk_002.csv
 - data/raw/week6/comments_chunk_003.csv
 - data/raw/week6/comments_chunk_004.csv


In [27]:
import glob
import pandas as pd
import os

chunk_paths = sorted(glob.glob("data/raw/week6/comments_chunk_*.csv"))
print("Chunks found:", len(chunk_paths))
print("\n".join(chunk_paths[:5]), "..." if len(chunk_paths) > 5 else "")

dfs = [pd.read_csv(p) for p in chunk_paths]
comments_raw = pd.concat(dfs, ignore_index=True)

print("Merged raw comments:", comments_raw.shape)
comments_raw.head()


Chunks found: 5
data/raw/week6/comments_chunk_000.csv
data/raw/week6/comments_chunk_001.csv
data/raw/week6/comments_chunk_002.csv
data/raw/week6/comments_chunk_003.csv
data/raw/week6/comments_chunk_004.csv 
Merged raw comments: (6727, 7)


,video_id,comment_id,text,like_count,published_at,channel,time_bin
0,iOTCsXd38Ng,UgxaVM4LSOrH4e0Pamt4AaABAg,Sand and Silica is are among the Most abundant...,1,2026-02-08T23:58:52Z,TED,2020+
1,iOTCsXd38Ng,UgywYl7CRxnk1yUoWCR4AaABAg,"Plastic is everywhere and I mean everywhere, ...",0,2026-02-08T23:00:34Z,TED,2020+
2,iOTCsXd38Ng,Ugyqmuz52AJOZBUTynd4AaABAg,Brava❤,1,2026-02-08T22:11:44Z,TED,2020+
3,iOTCsXd38Ng,UgzB3f0d76GDGn7uL6B4AaABAg,Amazing! My respects.,1,2026-02-08T22:02:43Z,TED,2020+
4,iOTCsXd38Ng,UgzLsOVSfvA8jzipqJ54AaABAg,IT IS A LIE! WE KNOW IT. DONT FOOL US ANY MORE...,3,2026-02-08T20:46:49Z,TED,2020+


In [28]:
print("Columns:", comments_raw.columns.tolist())
print("Null text:", comments_raw["text"].isna().sum() if "text" in comments_raw.columns else "NO text column found")

# Drop duplicates if you have comment_id
if "comment_id" in comments_raw.columns:
    before = len(comments_raw)
    comments_raw = comments_raw.drop_duplicates(subset=["comment_id"])
    print("Dropped duplicates:", before - len(comments_raw))

# Remove empty text
if "text" in comments_raw.columns:
    before = len(comments_raw)
    comments_raw = comments_raw[comments_raw["text"].astype(str).str.strip() != ""]
    print("Dropped empty text:", before - len(comments_raw))


Columns: ['video_id', 'comment_id', 'text', 'like_count', 'published_at', 'channel', 'time_bin']
Null text: 0
Dropped duplicates: 0
Dropped empty text: 0


In [29]:
os.makedirs("data/raw/week6", exist_ok=True)

merged_raw_path = "data/raw/week6/comments_raw_merged_batch001.csv"
comments_raw.to_csv(merged_raw_path, index=False)

print("Saved merged raw:", merged_raw_path)


Saved merged raw: data/raw/week6/comments_raw_merged_batch001.csv


In [30]:
os.makedirs("data/processed/week6", exist_ok=True)

processed_path = "data/processed/week6/comments_clean_batch001.csv"
comments_raw.to_csv(processed_path, index=False)

print("Saved processed batch:", processed_path)


Saved processed batch: data/processed/week6/comments_clean_batch001.csv


In [31]:
summary = {
    "n_chunks": len(chunk_paths),
    "n_rows_raw_merged": int(comments_raw.shape[0]),
}

# If you have video_id column
if "video_id" in comments_raw.columns:
    summary["n_unique_videos"] = int(comments_raw["video_id"].nunique())

# If you have channel column
if "channel" in comments_raw.columns:
    summary["channels"] = ", ".join(sorted(comments_raw["channel"].dropna().unique().tolist()))

summary_df = pd.DataFrame([summary])
summary_out = "data/processed/week6/batch001_summary.csv"
summary_df.to_csv(summary_out, index=False)

summary_df


,n_chunks,n_rows_raw_merged,n_unique_videos,channels
0,5,6727,125,TED


In [32]:
!ls data/raw/week6


comments_chunk_000.csv	comments_chunk_004.csv		  scrape_summary.csv
comments_chunk_001.csv	comments_raw_merged_batch001.csv  ted_tedx_videos.csv
comments_chunk_002.csv	comments_test_sample.csv
comments_chunk_003.csv	scrape_progress.csv


In [33]:
import os

os.makedirs("data/raw/week6", exist_ok=True)
os.makedirs("data/processed/week6", exist_ok=True)

print("Folders ensured:")
print(os.listdir("data"))


Folders ensured:
['data_dictionary.md', 'raw', 'video_list.csv', 'processed']


In [34]:
import glob
import pandas as pd

chunk_paths = sorted(glob.glob("data/raw/week6/comments_chunk_*.csv"))
print("Found chunks:", chunk_paths)

dfs = [pd.read_csv(p) for p in chunk_paths]
comments_raw = pd.concat(dfs, ignore_index=True)

merged_raw_path = "data/raw/week6/comments_raw_merged_batch001.csv"
comments_raw.to_csv(merged_raw_path, index=False)

print("Saved:", merged_raw_path)
print("Shape:", comments_raw.shape)


Found chunks: ['data/raw/week6/comments_chunk_000.csv', 'data/raw/week6/comments_chunk_001.csv', 'data/raw/week6/comments_chunk_002.csv', 'data/raw/week6/comments_chunk_003.csv', 'data/raw/week6/comments_chunk_004.csv']
Saved: data/raw/week6/comments_raw_merged_batch001.csv
Shape: (6727, 7)


In [35]:
processed_path = "data/processed/week6/comments_clean_batch001.csv"
comments_raw.to_csv(processed_path, index=False)

print("Saved:", processed_path)


Saved: data/processed/week6/comments_clean_batch001.csv


In [36]:
summary = {
    "n_rows": len(comments_raw),
    "n_videos": comments_raw["video_id"].nunique() if "video_id" in comments_raw.columns else None,
    "channels": ", ".join(sorted(comments_raw["channel"].unique()))
}

summary_df = pd.DataFrame([summary])
summary_path = "data/processed/week6/batch001_summary.csv"
summary_df.to_csv(summary_path, index=False)

summary_df


,n_rows,n_videos,channels
0,6727,125,TED


In [37]:
import os, glob

print("CWD:", os.getcwd())
print("Does data/processed/week6 exist?", os.path.isdir("data/processed/week6"))
print("What files are in data/processed/week6 right now?")
print(glob.glob("data/processed/week6/*"))


CWD: /content/SOSC314_Project/SOSC314_Project
Does data/processed/week6 exist? True
What files are in data/processed/week6 right now?
['data/processed/week6/batch001_summary.csv', 'data/processed/week6/ted_tedx_video_sampling_frame.csv', 'data/processed/week6/comments_clean_batch001.csv']


In [38]:
import glob

hits = glob.glob("**/comments_clean_batch001.csv", recursive=True) + glob.glob("**/batch001_summary.csv", recursive=True)
print("Found:", hits)


Found: ['data/processed/week6/comments_clean_batch001.csv', 'data/processed/week6/batch001_summary.csv']


In [39]:
import os, shutil

repo_root = "/content/SOSC314_Project"  # change ONLY if your repo folder name is different
src1 = "data/processed/week6/comments_clean_batch001.csv"
src2 = "data/processed/week6/batch001_summary.csv"

dst_dir = os.path.join(repo_root, "data/processed/week6")
os.makedirs(dst_dir, exist_ok=True)

# Move if they exist
for src in [src1, src2]:
    if os.path.exists(src):
        dst = os.path.join(dst_dir, os.path.basename(src))
        shutil.move(src, dst)
        print("Moved ->", dst)
    else:
        print("Not found at:", src)

print("Now in repo week6:", os.listdir(dst_dir))


Moved -> /content/SOSC314_Project/data/processed/week6/comments_clean_batch001.csv
Moved -> /content/SOSC314_Project/data/processed/week6/batch001_summary.csv
Now in repo week6: ['batch001_summary.csv', 'ted_tedx_video_sampling_frame.csv', 'comments_clean_batch001.csv', '.gitkeep']
